# 02 — Custom Experiment

How to register a new strategy and run the full pipeline.

Every experiment lives in `quant_lib/experiments/` as a small Python file that combines a `Hypothesis` with a `StrategyConfig` and calls `register()`.

## 1. Define your hypothesis

In [ ]:
from quant_lib.audit import Hypothesis, StrategyType
from quant_lib.experiments.base import (
    PeriodConfig, UniverseConfig, StrategyConfig, ExperimentConfig,
)
from quant_lib.experiments import register

# Hypothesis describes the EDGE you're testing. The narrative
# fields are required for the experiment journal.
hyp = Hypothesis(
    name="my_breakout_v1",
    strategy_type=StrategyType.VOL_COMPRESSION,
    mechanism="compression -> expansion breakout with volume",
    expectation="long when EMA200 > EMA200[1] AND price breaks 20-bar high",
    holding_period="1-3 days",
    exit_rules="trailing stop 3 ATR",
    invalidation="win rate < 40% over 30 OOS trades",
)

## 2. Build the config

In [ ]:
config = ExperimentConfig(
    name="my_breakout_v1",
    hypothesis=hyp,
    period=PeriodConfig(
        training=("2020-01-01", "2024-12-31"),
        holdout=("2025-01-01", "2025-06-30"),
    ),
    universe=UniverseConfig(
        symbols=["BTCUSDT", "ETHUSDT", "SOLUSDT"],
        min_volume_usd=10_000_000,
    ),
    strategy=StrategyConfig(
        leverage=3.0,
        pf_weight_clamp_floor=0.5,
        pf_weight_clamp_ceiling=1.5,
    ),
)

## 3. Register and run

In [ ]:
register(config)
result = quant_lib.run_explore(
    experiment_name="my_breakout_v1",
    cache_dir="./data_cache",
)

## 4. Inspect & commit (Phase 4)

If the SPA p-value is below your threshold and you want to test the strategy against the holdout:

```python
if result.spa_p_value < 0.05:
    commit_result = quant_lib.run_commit(
        experiment_name="my_breakout_v1",
        cache_dir="./data_cache",
    )
    print(f"Holdout PSR: {commit_result.holdout_psr:.4f}")
```

**WARNING:** `run_commit` is IRREVERSIBLE. The holdout seal is broken once you commit; subsequent runs cannot re-test against the same holdout.